# 🔧 Notebook 04 — Data Preprocessing

---

## Project Context

| Field | Detail |
|---|---|
| **Competition** | SATRIA DATA 2026 — Big Data Challenge |
| **Experiment** | Baseline CNN — Experiment 01 |
| **Stage** | Preprocessing (Step 2 of 7) |

---

## Tujuan Tahap Preprocessing

Tahap ini bertugas mengubah data mentah menjadi format yang siap dikonsumsi model:
1. Scanning gambar corrupt (dari EDA)
2. Melakukan Stratified Split (80% Train, 20% Val)
3. Menetapkan Transformasi (Resize, Crop, Normalize, ColorJitter)
4. Menggunakan WeightedRandomSampler (untuk mengatasi imbalanced dataset)
5. Menghasilkan DataLoader yang efisien

> **Catatan:** Sesuai `CLAUDE.md`, notebook ini hanya berfungsi sebagai antarmuka (orchestrator). Seluruh implementasi teknis berada di modul `src/preprocessing/`.

In [5]:
# ============================================================
# CELL 1 — IMPORTS & SETUP
# ============================================================
import sys
import logging
from pathlib import Path
import yaml

# Tambahkan root proyek ke sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
from src.preprocessing.splitter import collect_dataset, stratified_split, save_split_report, get_split_summary
from src.preprocessing.dataloader import build_dataloaders, verify_dataloader

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

with open(PROJECT_ROOT / 'configs' / 'baseline.yaml', 'r') as f:
    config = yaml.safe_load(f)

SEED = config['experiment']['seed']
set_seed(SEED)
logger.info(f"Seed set to {SEED}")

22:44:08 | Seed set to 42


Seed set to: 42


---
## Step 1 — Collect Data & Stratified Split
Kami membaca seluruh direktori data dan membaginya 80/20 secara *stratified in-memory* (tanpa menyalin file ke disk).

In [6]:
# ============================================================
# CELL 2 — DATA SPLITTING
# ============================================================
data_root = PROJECT_ROOT / config['data']['raw_dir']
train_dir = data_root / config['data']['train_subdir']

df_all = collect_dataset(train_dir)
logger.info(f"Total data terkumpul: {len(df_all)}")

df_train, df_val = stratified_split(
    df_all, 
    train_ratio=config['split']['train_ratio'], 
    val_ratio=config['split']['val_ratio'], 
    random_seed=SEED
)

reports_dir = PROJECT_ROOT / config['output']['reports_dir']
save_split_report(df_train, df_val, reports_dir)

print("\nRingkasan Split:")
print(get_split_summary(df_train, df_val))

22:44:08 | Kelas terdeteksi: ['0_Recyclable', '1_Electronic', '2_Organic']
22:44:08 | Total gambar dikumpulkan: 26,524
22:44:08 | Total data terkumpul: 26524
22:44:08 | Split selesai (seed=42):
22:44:08 |   Train : 21,219 gambar (80%)
22:44:08 |   Val   : 5,305 gambar (20%)
22:44:08 |   Train distribution: {'Organic': 10054, 'Recyclable': 7999, 'Electronic': 3166}
22:44:08 |   Val distribution: {'Organic': 2513, 'Recyclable': 2000, 'Electronic': 792}
22:44:08 | Saved: D:\Data Analysis\Smart Waste Classification\outputs\reports\preprocessing_split.csv
22:44:08 | Saved: D:\Data Analysis\Smart Waste Classification\outputs\reports\preprocessing_split_stats.json



Ringkasan Split:
  DATASET SPLIT SUMMARY
  Total gambar      : 26,524
  Training set      : 21,219 (80.0%)
  Validation set    : 5,305 (20.0%)

  Training Distribution:
    Organic       : 10,054 (47.4%)
    Recyclable    : 7,999 (37.7%)
    Electronic    : 3,166 (14.9%)
  Validation Distribution:
    Organic       : 2,513 (47.4%)
    Recyclable    : 2,000 (37.7%)
    Electronic    : 792 (14.9%)


---
## Step 2 — Build DataLoaders
DataLoader mengonversi DataFrame ke tensor iterator. `train_loader` otomatis dilengkapi dengan `WeightedRandomSampler` untuk mengatasi ketidakseimbangan kelas.

In [7]:
# ============================================================
# CELL 3 — DATALOADER INITIALIZATION
# ============================================================
train_loader, val_loader = build_dataloaders(
    df_train=df_train,
    df_val=df_val,
    config=config,
    use_weighted_sampler=True
)

print("\nVerifikasi Train Loader:")
verify_dataloader(train_loader, split_name='train', n_batches=1)

print("\nVerifikasi Val Loader:")
verify_dataloader(val_loader, split_name='val', n_batches=1)

22:44:09 | WasteDataset [train] dibuat: 21,219 gambar, transform=Yes
22:44:09 | WeightedRandomSampler diaktifkan untuk training.
22:44:09 | Train DataLoader: 21,219 gambar, batch_size=32, num_batches=663
22:44:09 | WasteDataset [val] dibuat: 5,305 gambar, transform=Yes
22:44:09 | Val DataLoader: 5,305 gambar, batch_size=32, num_batches=166
c:\Users\Microsoft\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Verifikasi Train Loader:


22:44:09 | [train] Batch 0: shape=(32, 3, 224, 224), range=[-2.118, 2.640]
22:44:10 | [val] Batch 0: shape=(32, 3, 224, 224), range=[-2.118, 2.640]



Verifikasi Val Loader:


{'batch_index': 0,
 'split': 'val',
 'images_shape': (32, 3, 224, 224),
 'labels_shape': (32,),
 'dtype': 'torch.float32',
 'pixel_min': -2.1179,
 'pixel_max': 2.64,
 'pixel_mean': 0.5607,
 'unique_labels': [0, 1, 2]}